In [1]:
%cd /Users/amarmesic/Documents/tudelft/thesis/DNANet

/Users/amarmesic/Documents/tudelft/thesis/DNANet


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Test Baseline Threshold Segmentation Model

This notebook demonstrates how to load a HID dataset, apply the baseline threshold segmentation model, and inspect predictions.

## Tracking Performance of different thresholds

### 70 - Best Pixel F1
- Pixel F1 Score: 0.5741
- Pixel Precision: 0.5376
- Pixel Recall: 0.6159
- Allele F1 Score: 0.771
- Allele Precision: 0.7618
- Allele Recall: 0.7805

### 75 - Best Allele F1
- Pixel F1 Score: 0.5735
- Pixel Precision: 0.5425
- Pixel Recall: 0.6082
- Allele F1 Score: 0.7738
- Allele Precision: 0.7726
- Allele Recall: 0.7749

### 100
- Pixel F1 Score: 0.561
- Pixel Precision: 0.5737
- Pixel Recall: 0.5488
- Allele F1 Score: 0.7699
- Allele Precision: 0.8186
- Allele Recall: 0.7267

## Visualize Example Prediction

Let's visualize the segmentation mask for the first image in the dataset.

In [3]:
import neptune
from config_io import load_dataset
from DNAnet.models.segmentation.threshold import ThresholdSegmentationModel
from DNAnet.evaluation.segmentation.allele_metrics import allele_f1_score, allele_precision, allele_recall
from DNAnet.evaluation.segmentation.pixel_metrics import pixel_f1_score, pixel_precision, pixel_recall

# Define genotype folds
genotype_folds = [
    [30, 31, 32, 33, 34],
    [35, 36, 37, 38, 39],
    [40, 41, 42, 43],
    [44, 45, 46, 47],
    [29, 48, 49, 50]
]

# Load dataset once
dataset = load_dataset("provedit_data.yaml")

for f_index, fold in enumerate(genotype_folds):
    # Split test set for this fold
    test_set, _ = dataset.split_by_genotypes(set(fold))
    
    # Initialize the baseline thresholding model
    model = ThresholdSegmentationModel(threshold=75, apply_allele_caller=True)
    
    # Make predictions
    predictions = [model.predict(image) for image in test_set]
    
    # Compute metrics
    f1 = pixel_f1_score(test_set, predictions)
    precision = pixel_precision(test_set, predictions)
    recall = pixel_recall(test_set, predictions)
    allele_f1 = allele_f1_score(test_set, predictions)
    allele_prec = allele_precision(test_set, predictions)
    allele_rec = allele_recall(test_set, predictions)
    
    # Start Neptune run
    run = neptune.init_run(
        name=f"基线-75rfu-DNANet的长-1正真:0正假-预：缩放-折{f_index}",
        project="amar-mesic/dna-thesis",
        api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
    )
    
    # Log metadata
    meta = {
        "experiment": "AT_Baseline",
        "model": "AT_75",
        "dataset": "proved_it",
        "fold": f_index,
        "seed": 0,
    }
    for k, v in meta.items():
        run[f'meta/{k}'] = v
    
    # Log metrics
    run["test/pixel_f1"] = float(f"{f1:.4g}")
    run["test/pixel_precision"] = float(f"{precision:.4g}")
    run["test/pixel_recall"] = float(f"{recall:.4g}")
    run["test/allele_f1"] = float(f"{allele_f1:.4g}")
    run["test/allele_precision"] = float(f"{allele_prec:.4g}")
    run["test/allele_recall"] = float(f"{allele_rec:.4g}")
    
    run.stop()

2025-10-03 17:40:32 INFO     Found 675 files in /Users/amarmesic/Documents/tudelft/thesis/datasets/USE THIS - PROVEDIt_2-5-Person Profiles_3500 5sec_GF29cycles


2025-10-03 17:40:33 WARNING  Size standard for E04_RD14-0003-33_34-1,2-M4d-0.045GF-Q1.3_05.5sec.hid differs 18.912544484957948 from the expected 
2025-10-03 17:40:35 WARNING  Size standard for H08_RD14-0003-49_50_29-1,4,1-M2e-0.378GF-Q1.4_08.5sec.hid differs 13.401428160223645 from the expected 
2025-10-03 17:40:37 WARNING  Size standard for G03_RD14-0003-40_41-1,4-M3e-0.625GF-Q0.8_07.5sec.hid differs 12.982847650856456 from the expected 
2025-10-03 17:40:39 WARNING  Size standard for E03_RD14-0003-48_49_50_29-1,4,4,4-M2I15-0.75GF-Q1.1_05.5sec.hid differs 13.001695302289932 from the expected 
2025-10-03 17:40:40 WARNING  Size standard for E05_RD14-0003-46_47_48-1,1,1-M3I35-0.189GF-QLAND_05.5sec.hid differs 18.824515396807158 from the expected 
2025-10-03 17:40:46 INFO     Number of valid images: 670
2025-10-03 17:40:46 INFO     Number of files limited to: 670
2025-10-03 17:41:12 INFO     ✅ Community A: 96 images
2025-10-03 17:41:12 INFO     ✅ Community B: 469 images
2025-10-03 17:41:12

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-721
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 11 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 11 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-721/metadata


2025-10-03 17:41:14 INFO     ✅ Community A: 66 images
2025-10-03 17:41:14 INFO     ✅ Community B: 469 images
2025-10-03 17:41:14 INFO     ⚠️ Ambiguous: 135 images discarded due to overlap


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-722
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 11 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 11 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-722/metadata


...

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-723
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 11 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 11 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-723/metadata


2025-10-03 17:41:16 INFO     ✅ Community A: 83 images
2025-10-03 17:41:16 INFO     ✅ Community B: 523 images
2025-10-03 17:41:16 INFO     ⚠️ Ambiguous: 64 images discarded due to overlap


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-724
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 11 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 11 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-724/metadata


2025-10-03 17:41:18 INFO     ✅ Community A: 82 images
2025-10-03 17:41:18 INFO     ✅ Community B: 492 images
2025-10-03 17:41:18 INFO     ⚠️ Ambiguous: 96 images discarded due to overlap
2025-10-03 17:41:18 WARNING  No predictions present in dye row 0
2025-10-03 17:41:18 WARNING  No predictions present in dye row 2
2025-10-03 17:41:18 WARNING  No predictions present in dye row 3


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-725
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 11 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 11 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-725/metadata
